In [1]:
import pandas as pd

from ada_fsaudit_bridge import (
    att_sample,
    configure_environment,
    cvs_sample,
    load_dataset,
    lower_bound,
    mus_sample,
    set_notebook_context,
    upper_bound,
)
from scipy.stats import hypergeom

ADA_WORKSHOP_ID = "auxiliary-variables-and-stratification"
ADA_CHAPTER = "3"
ADA_NOTEBOOK_SOURCE = "notebooks/support/auxiliary-variables-and-stratification/support.Rmd"

def ada_set_context(exercise_ref: str) -> None:
    set_notebook_context(
        chapter=ADA_CHAPTER,
        exercise=exercise_ref,
        notebook=f'{ADA_WORKSHOP_ID}:{ADA_NOTEBOOK_SOURCE}',
    )

configure_environment()


{'r_version': '4.3.1',
 'fsaudit_version': '0.3.4',
 'library_paths': ['/Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/library'],
 'configured_seed': None,
 'context': {},
 'bridge_version': '0.1.0'}

## Exercise 3.1. The `inventoryData` file


Details of the data file can be obtained by inspecting the dataset columns and
row count.


In [2]:
from IPython.display import display
ada_set_context("3.1")
inventoryData = load_dataset("inventoryData")
inventoryData.columns.tolist()
len(inventoryData)
display(inventoryData["bv"].sum())
display(inventoryData.iloc[:6, :3])


np.float64(7360816.0)

,item,bv,av
1,1,120.60,109.92
2,2,2600.10,2219.24
3,3,4899.90,6398.96
4,4,1660.49,1676.29
5,5,5398.28,4511.42
6,6,1063.70,953.08


## Exercise 3.2. Working with the CVS object


We first create a Classical Variables Sampling ("CVS") object. An object is a container that gathers all the data relevant to a particular sampling application. The `FSaudit` package recognizes three such objects. We work with the CVS object `cvs_obj` in this chapter, and with the attribute `att_obj` and MUS `mus_obj` objects in Chapter 5.
We create the CVS object `mySample` by running the `cvs_obj` function, and at the same time fill it with information on the required sample size `n`, the relevant book values `bv`, identifiers `id`, and the random seed number `seed`.


In [3]:
ada_set_context("3.2")
mySample = cvs_sample(
    n=400,
    bv=inventoryData["bv"],
    id=inventoryData["item"],
    seed=12345,
)


The CVS object has 19 different attributes attached to it, that will be filled as we proceed from sample planning, through stratification and selection to evaluation. The following is a list of all attributes.


In [4]:
ada_set_context("3.2")
mySample.field_names()


['id',
 'bv',
 'popn',
 'popBv',
 'n',
 'strata',
 'classes',
 'stratMeth',
 'classSumm',
 'stratSumm',
 'strat',
 'cl',
 'desPrec',
 'sdAv',
 'alloc',
 'seed',
 'sample',
 'av',
 'results']

Some of these attributes are empty for now, for example `sample`.


In [5]:
ada_set_context("3.2")
mySample.sample


In order to replicate the experiment in the book, you need to use the same
random number seed as we did.


In [6]:
ada_set_context("3.2")
configure_environment(seed=12345)


{'r_version': '4.3.1',
 'fsaudit_version': '0.3.4',
 'library_paths': ['/Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/library'],
 'configured_seed': 12345,
 'context': {'chapter': '3',
  'exercise': '3.2',
  'notebook': 'auxiliary-variables-and-stratification:notebooks/support/auxiliary-variables-and-stratification/support.Rmd'},
 'bridge_version': '0.1.0'}

We sample 400 units from the `inventoryData` dataset with the `select()` function. This effectively fills the `sample` attribute in `mySample`.


In [7]:
ada_set_context("3.2")
mySample.select()
mySample.sample.head()


,item,bv,strat
2190,2190,5103.93,1.0
51,51,212.12,1.0
720,720,186.85,1.0
730,730,39.92,1.0
1244,1244,666.61,1.0


We audit the sampling units selected, and submit the audit values to the CVS object for the sample to be evaluated.


In [8]:
ada_set_context("3.2")
audit_values = inventoryData.set_index("item").loc[mySample.sample["item"], "av"]
mySample.evaluate(av=audit_values)


The evaluation results are stored in the `evalResults` attribute of the CVS object. This attribute itself is a list of 17 different attributes.


In [9]:
ada_set_context("3.2")
list(mySample.eval_results.keys())


['Estimates',
 'Mean per unit estimation',
 'Most likely total audited amount mpu',
 'Most likely total error mpu',
 'Achieved precision mpu',
 'Difference estimation',
 'Most likely total audited amount difference',
 'Most likely total error difference',
 'Achieved precision difference',
 'Ratio estimation',
 'Most likely total audited amount ratio',
 'Most likely total error ratio',
 'Achieved precision ratio',
 'Regression estimation',
 'Most likely total audited amount regression',
 'Most likely total error regression',
 'Achieved precision regression']

## Exercise 3.3. Mean-per-unit estimator


In Section 4.1 we found an estimate of the population value $\hat{Y}_{MPU}$ = 7,561,859, and achieved precision of 684,415. These values are stored in the attribute `evalResults` of `mySample`.


In [10]:
ada_set_context("3.3")
mySample.eval_results["Most likely total audited amount mpu"]


[7106461.949999998]

In [11]:
ada_set_context("3.3")
mySample.eval_results["Achieved precision mpu"]


[667833.918671754]

The prediction interval is stored in:


In [12]:
ada_set_context("3.3")
mySample.eval_results["Estimates"].iloc[3:5, 0]


lower bound    6.438628e+06
upper bound    7.774296e+06
Name: mpu, dtype: float64

In this call, the numbers in brackets refer to the 4th and 5th row of the Estimates table. The first column stores the results of the mean-per-unit estimator.
Alternatively, we can refer to the label of the relevant column.


In [13]:
ada_set_context("3.3")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("mpu")]


lower bound    6.438628e+06
upper bound    7.774296e+06
Name: mpu, dtype: float64

## Exercise 3.4. The Regression Estimator


The results for the regression estimator are stored in a similar way.


In [14]:
ada_set_context("3.4")
mySample.eval_results["Most likely total audited amount regression"]


[7311522.727831856]

In [15]:
ada_set_context("3.4")
mySample.eval_results["Achieved precision regression"]


[204137.43846585084]

The prediction interval is determined by:


In [16]:
ada_set_context("3.4")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("regr")]


lower bound    7.107385e+06
upper bound    7.515660e+06
Name: regr, dtype: float64

## Exercise 3.5. The Difference Estimator


The results for the difference estimator are as follows.


In [17]:
ada_set_context("3.5")
mySample.eval_results["Most likely total audited amount difference"]


[7312970.562499968]

In [18]:
ada_set_context("3.5")
mySample.eval_results["Achieved precision difference"]


[203930.9046143157]

The prediction interval is determined by:


In [19]:
ada_set_context("3.5")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("diff")]


lower bound    7.109040e+06
upper bound    7.516901e+06
Name: diff, dtype: float64

## Exercise 3.6. The Ratio Estimator


Finally, the results of the ratio estimator.


In [20]:
ada_set_context("3.6")
mySample.eval_results["Most likely total audited amount ratio"]


[7311589.5070913285]

In [21]:
ada_set_context("3.6")
mySample.eval_results["Achieved precision ratio"]


[203881.57184158903]

The prediction interval is determined by:


In [22]:
ada_set_context("3.6")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("ratio")]


lower bound    7.107708e+06
upper bound    7.515471e+06
Name: ratio, dtype: float64

An overview of all estimates is given by


In [23]:
ada_set_context("3.6")
mySample.eval_results["Estimates"]


,mpu,diff,ratio,regr
audit value,7.106462e+06,7.312971e+06,7.311590e+06,7.311523e+06
misstatement,2.543540e+05,4.784544e+04,4.922649e+04,4.929327e+04
precision,6.678339e+05,2.039309e+05,2.038816e+05,2.041374e+05
lower bound,6.438628e+06,7.109040e+06,7.107708e+06,7.107385e+06
upper bound,7.774296e+06,7.516901e+06,7.515471e+06,7.515660e+06
effective df,3.990000e+02,3.990000e+02,3.990000e+02,3.990000e+02


## Exercise 3.7. Using CVS with sporadic errrors


We start with setting up a cvs_obj object, and fill it with the parameters. We use the prefix `ar` for accounts receivable.


In [24]:
ada_set_context("3.7")
accounts_receivable = load_dataset("accounts_receivable")
arSample = cvs_sample(
    bv=accounts_receivable["amount"],
    id=accounts_receivable["invoice"],
    n=100,
)


We select the sample, using a seed equal to 1.


In [25]:
ada_set_context("3.7")
arSample.select(seed=1)


We look up the audit values of the selected sampling units.


In [26]:
ada_set_context("3.7")
audit_values = accounts_receivable.set_index("invoice").loc[arSample.sample["item"], "av2"]


We then evaluate the sample and look at the `Estimates` attribute.


In [27]:
ada_set_context("3.7")
arSample.evaluate(av=audit_values)
arSample.eval_results["Estimates"]


,mpu,diff,ratio,regr
audit value,1.436194e+07,1.317729e+07,1.320332e+07,1.317923e+07
misstatement,-8.619400e+05,3.227140e+05,2.966797e+05,3.207673e+05
precision,2.379609e+06,6.744397e+05,6.965393e+05,6.744241e+05
lower bound,1.198233e+07,1.250285e+07,1.250678e+07,1.250481e+07
upper bound,1.674155e+07,1.385173e+07,1.389986e+07,1.385366e+07
effective df,9.900000e+01,7.000000e+00,7.000000e+00,7.000000e+00


The number of differences is stored in the `#_Errors` attribute, under each of the estimation methods. This way, you can tell if precision was calculated with the sporadic error (k = 4 to 19) method, or using the standard CVS method.


In [28]:
ada_set_context("3.7")
arSample.eval_results["Regression estimation"]["#_Errors"]


1    8
Name: #_Errors, dtype: int32

Most of the calculations presented in Section 4.7 are easy to follow, so we will not repeat them all in this workshop. Instead, we zoom in on some specific parts of the overall calculations.
The value of $M_U$ that is used in, for example, Equation 4.23 is obtained as follows.


In [29]:
ada_set_context("3.7")
N = len(accounts_receivable)
m = 6
(m_u := upper_bound(k=m, popn=N, n=100, alpha=0.05) / N)


0.1147

The effective degrees of freedom is $m - 1$. This explains the $t$ value used when calculating the confidence intervals.


In [30]:
from scipy.stats import t
ada_set_context("3.7")
t.ppf(0.975, df=m - 1)


np.float64(2.570581835636314)

## Exercise 3.8. Stratification with equal recorded boundaries


We start with creating a CVS object.


In [31]:
ada_set_context("3.8")
equal = cvs_sample(bv=inventoryData["bv"], id=inventoryData["item"])


We then stratify with the equal method (`stratMeth = equal`), and view the summary.


In [32]:
ada_set_context("3.8")
equal.stratify(strata=3, stratMeth="equal")
equal.field("stratSumm")


,strat,cum_bv,freq,minBv,maxBv
1,1,2452927.31,2431,1.57,2478.93
2,2,2453454.96,710,2480.29,4833.28
3,3,2454433.73,359,4839.62,18496.21


## Exercise 3.9. Stratification with the cumulative method


In [33]:
ada_set_context("3.9")
cumul = cvs_sample(bv=inventoryData["bv"], id=inventoryData["item"])
cumul.stratify(strata=3, classes=10, stratMeth="cumulative")


The classification in Table 4.6 is stored in the `classSum` attribute.


In [34]:
ada_set_context("3.9")
cumul.field("classSumm")


,cl,freq,minBv,maxBv,sqrtFreq,cmSum,strat
1,1.0,2030.0,1.57,1848.60,45.055521,45.055521,1.0
2,2.0,857.0,1851.65,3694.56,29.274562,74.330084,2.0
3,3.0,360.0,3700.70,5547.71,18.973666,93.303750,2.0
4,4.0,154.0,5556.67,7398.87,12.409674,105.713423,3.0
5,5.0,63.0,7406.10,9238.08,7.937254,113.650677,3.0
6,6.0,24.0,9273.30,11077.31,4.898979,118.549657,3.0
7,7.0,7.0,11299.53,12723.03,2.645751,121.195408,3.0
8,8.0,4.0,13131.76,14125.05,2.000000,123.195408,3.0
9,10.0,1.0,18496.21,18496.21,1.000000,124.195408,3.0


The stratification summary is then:


In [35]:
ada_set_context("3.9")
cumul.field("stratSumm")


,strat,minBv,maxBv,freq,cumSum
1,1,1.57,1848.60,2030.0,45.055521
2,2,1851.65,5547.71,1217.0,48.248228
3,3,5556.67,18496.21,253.0,30.891658


We calculate the required sample size for a desired precision of 200,000.


In [36]:
ada_set_context("3.9")
cumul.size(desPrec=200000).n


[584.0]

Select sample


In [37]:
ada_set_context("3.9")
cumul.select(seed=12345)
cumul.sample.head()


,item,bv,strat
142,2104,86.81,1
51,590,30.15,1
720,2080,477.55,1
730,1274,484.90,1
1244,783,929.81,1


Obtain the audit values and evaluate the sample.


In [38]:
ada_set_context("3.9")
true_values = inventoryData.set_index("item").loc[cumul.sample["item"], "av"]
cumul.evaluate(av=true_values)


A summary table of the estimates for each of the estimation methods is stored in the `Estimates` argument, which is part of the `evalResults` argument.


In [39]:
ada_set_context("3.9")
cumul.eval_results["Estimates"]


,mpu,diff,ratio,regr
audit value,7.480442e+06,7.460689e+06,7.460422e+06,7.460333e+06
misstatement,-1.196260e+05,-9.987310e+04,-9.960581e+04,-9.951689e+04
precision,2.388348e+05,1.251972e+05,1.251485e+05,1.254901e+05
lower bound,7.241607e+06,7.335492e+06,7.335273e+06,7.334843e+06
upper bound,7.719277e+06,7.585886e+06,7.585570e+06,7.585823e+06
effective df,5.760000e+02,4.640000e+02,4.640000e+02,4.640000e+02
